In [1]:
!pip install Web3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.6/306.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 22.7 MB/s eta 0:00:00


In [2]:
!pip install eth_tester

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 4.7 MB/s eta 0:00:00


In [3]:
!pip install "web3[tester]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.7/827.7 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 2.4 MB/s eta 0:00:00
  Created wheel for safe-pysha3: filename=safe_pysha3-1.0.4-cp311-cp311-linux_x86_64.whl size=147640 sha256=ed0970d5145582e60d80434cbf472fb9607d0e86c6d3bf35dbb661f43d096ff6
  Stored in directory: /root/.cache/pip/wheels/b3/ef/41/7bba4fbb915ace87eea10e6a287a991a7ced54d98726906766
Successfully built safe-pysha3


In [4]:
!pip install py-solc-x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.31.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [5]:
import hashlib
import json
import time
import pandas as pd
from cryptography.fernet import Fernet

# Generate/load symmetric encryption key
try:
    with open("secret.key", "rb") as f:
        key = f.read()
except FileNotFoundError:
    key = Fernet.generate_key()
    with open("secret.key", "wb") as f:
        f.write(key)

fernet = Fernet(key)

class Block:
    def __init__(self, index, timestamp, encrypted_data, previous_hash):
        self.index = index
        self.timestamp = timestamp
        self.encrypted_data = encrypted_data  # bytes
        self.previous_hash = previous_hash
        self.hash = self.compute_hash()

    def compute_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "encrypted_data": self.encrypted_data.decode(),
            "previous_hash": self.previous_hash
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, time.time(), fernet.encrypt(b"Genesis Block"), "0")

    def add_block(self, data_dict):
        data_str = json.dumps(data_dict)
        encrypted_data = fernet.encrypt(data_str.encode())
        last_block = self.chain[-1]
        new_block = Block(len(self.chain), time.time(), encrypted_data, last_block.hash)
        self.chain.append(new_block)

    def display_chain(self):
        for block in self.chain:
            print(f"Index: {block.index}")
            print(f"Timestamp: {block.timestamp}")
            print(f"Encrypted Data (base64): {block.encrypted_data}")
            print(f"Decrypted Data: {fernet.decrypt(block.encrypted_data).decode(errors='ignore')}")
            print(f"Previous Hash: {block.previous_hash}")
            print(f"Hash: {block.hash}")
            print("-" * 50)

# Example: Store DataFrame in Blockchain
def store_dataframe(df, blockchain):
    for _, row in df.iterrows():
        blockchain.add_block(row.to_dict())

# Example usage
if __name__ == "__main__":
    # Create a sample DataFrame
    data = {
        "name": ["Alice", "Bob", "Charlie"],
        "age": [30, 25, 35],
        "medical_record": ["O+", "A-", "B+"]
    }
    df = pd.DataFrame(data)

    # Initialize and populate blockchain
    blockchain = Blockchain()
    store_dataframe(df, blockchain)
    blockchain.display_chain()


Index: 0
Timestamp: 1746087654.4507039
Encrypted Data (base64): b'gAAAAABoEy7m3ofa4xOySQS8osFrA0Kvn5h_IiKtTWHsoMTgU_ZdLFAglg8Vb9E8NeFmVjrwAQm6eSJ-aZ6_R3CNu9U0i5oaIA=='
Decrypted Data: Genesis Block
Previous Hash: 0
Hash: 7ce67e6321059f7ba382aecc223717deed7a213f60753a4e3a2c2d6d2635144e
--------------------------------------------------
Index: 1
Timestamp: 1746087654.4591088
Encrypted Data (base64): b'gAAAAABoEy7mGaAuWNHEf-EQeyoQJ1rrA2FU0AYpntVBlsh3RoWINDp1ySrBe82ED0yFP9WojI6S2L_a-wdTFotO9j7OL1iObmxrxZ6oLe-wSEdIyr0xYdg6vfNm9hyPciRg_UcuxN0HU-FrG5c-WH_pZ-9-8vFYeA=='
Decrypted Data: {"name": "Alice", "age": 30, "medical_record": "O+"}
Previous Hash: 7ce67e6321059f7ba382aecc223717deed7a213f60753a4e3a2c2d6d2635144e
Hash: 1831ce96ce0e448310cf4c715b25752fd8271529d13e65879979acee5e2cfd5f
--------------------------------------------------
Index: 2
Timestamp: 1746087654.4595733
Encrypted Data (base64): b'gAAAAABoEy7mOe2gLIqT2UmEiuSaPraIxWGaPfJkl9pU96gtjcwHowHWjx2zw21cQoZ_P66MSyCduZgckao89zqB0UQNcE

In [6]:
import pandas as pd
import numpy as np
import json
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from flask import Flask, Response, request, jsonify
from web3 import Web3, HTTPProvider, EthereumTesterProvider
from solcx import install_solc, compile_source

install_solc(version='latest')

<Version('0.8.29')>

In [7]:
!git clone https://github.com/digvijaysingh1707/Blockchain_based_Federated_learning.git

Cloning into 'Blockchain_based_Federated_learning'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 19.27 KiB | 6.42 MiB/s, done.


In [8]:
df = pd.read_csv('Blockchain_based_Federated_learning/uci_malware_detection.csv')

In [9]:

y = df["Label"]
X = df.drop("Label", axis=1)

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2, random_state=42)

In [10]:
X_train

,F_1,F_2,F_3,F_4,F_5,F_6,F_7,F_8,F_9,F_10,...,F_522,F_523,F_524,F_525,F_526,F_527,F_528,F_529,F_530,F_531
192,1,0,1,0,1,0,1,0,1,0,...,1,0,0,0,1,0,1,0,1,0
75,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,1,0,1,0,1,0
84,1,0,1,0,1,0,1,0,1,0,...,0,0,0,1,0,1,0,0,1,0
361,1,0,1,0,1,0,1,0,1,0,...,0,0,1,0,1,0,0,0,1,0
16,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
106,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
270,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,1,0,0,0,1,0
348,1,0,1,0,1,0,1,0,1,0,...,0,0,0,0,1,0,0,0,1,0


In [11]:
w3 = Web3(EthereumTesterProvider())
w3.is_connected()

<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()


True

In [12]:
compiled_sol = compile_source(
    '''
    pragma solidity >0.5.0;

    contract ModelParameters {
        string public encoded;

        constructor() public {
            encoded = '';
        }

        function setModelParameters(string memory _encoded) public {
            encoded = _encoded;
        }

        function getModelParameters() view public returns (string memory) {
            return encoded;
        }
    }
    ''',
    output_values=['abi', 'bin']
)

contract_id, contract_interface = compiled_sol.popitem()
bytecode = contract_interface['bin']
abi = contract_interface['abi']

ModelParameters = w3.eth.contract(abi=abi, bytecode=bytecode)

tx_hash = ModelParameters.constructor().transact()
tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

ml_contract = w3.eth.contract(
    address=tx_receipt.contractAddress,
    abi=abi
)

In [13]:
cls = LogisticRegression()
cls.fit(X_train,y_train)

<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()


LogisticRegression()

In [14]:
predictions = cls.predict(X_test);

accuracy = np.sum(predictions == y_test) / y_test.shape[0] * 100
conf_matrix = confusion_matrix(predictions, y_test)
precision = conf_matrix[0,0] / (conf_matrix[0,0] + conf_matrix[0,1]) * 100

print(conf_matrix)
print("Accuracy: {0:.6f}%".format(accuracy))
print("Precision: {0:.6f}%".format(precision))

[[57  1]
 [ 1 16]]
Accuracy: 97.333333%
Precision: 98.275862%


In [15]:
encoded = json.dumps((cls.coef_.tolist(), cls.intercept_.tolist(), cls.classes_.tolist()))
encoded

'[[[-0.002654774890015878, 0.0, -0.0052246612138527205, 0.002569886323836836, -0.0052246612138527205, 0.0, -0.0052246612138527205, 0.0, -0.0052246612138527205, 0.0, -0.0748442132326684, 0.0, 0.06647560919989288, 0.0, -0.06185413517636361, 0.0, 0.005063296884227979, 0.0, 0.28614423329762173, -0.39175684738651567, 0.1539673026842246, 0.0, 0.06738652803833539, 0.08658077464588923, 0.06738652803833539, -0.09570850993860179, 0.06738652803833539, 0.0, 0.1580485870321436, -0.1607033619221595, 0.1539673026842246, 0.0, 0.0771237519035868, 0.0, 0.00642145966599515, 0.09192105836478325, -0.09714571957863588, 0.09192105836478325, -0.09714571957863588, 4.878774934525946e-05, 0.06733774028899014, -0.10703616915963382, 0.06733774028899014, -0.07256240150284274, 0.06733774028899014, 0.07986251398318893, 0.18629698250443144, 0.02423772932974272, -0.2671520499623872, 0.26192738874853455, -0.2838883984454118, 0.179065986029245, -0.2647794524287589, 4.878774934525946e-05, 0.1865741682719344, 0.03091746833

In [16]:
w3.eth.defaultAccount = w3.eth.accounts[0]

tx_hash = ml_contract.functions.setModelParameters(encoded).transact()

tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

<frozen importlib._bootstrap>:1047: ImportWarning: _PyDriveImportHook.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:1047: ImportWarning: _BokehImportHook.find_spec() not found; falling back to find_module()


In [17]:
w3.eth.defaultAccount = w3.eth.accounts[1]

encoded_parameters = ml_contract.functions.getModelParameters().call()
encoded_parameters

'[[[-0.002654774890015878, 0.0, -0.0052246612138527205, 0.002569886323836836, -0.0052246612138527205, 0.0, -0.0052246612138527205, 0.0, -0.0052246612138527205, 0.0, -0.0748442132326684, 0.0, 0.06647560919989288, 0.0, -0.06185413517636361, 0.0, 0.005063296884227979, 0.0, 0.28614423329762173, -0.39175684738651567, 0.1539673026842246, 0.0, 0.06738652803833539, 0.08658077464588923, 0.06738652803833539, -0.09570850993860179, 0.06738652803833539, 0.0, 0.1580485870321436, -0.1607033619221595, 0.1539673026842246, 0.0, 0.0771237519035868, 0.0, 0.00642145966599515, 0.09192105836478325, -0.09714571957863588, 0.09192105836478325, -0.09714571957863588, 4.878774934525946e-05, 0.06733774028899014, -0.10703616915963382, 0.06733774028899014, -0.07256240150284274, 0.06733774028899014, 0.07986251398318893, 0.18629698250443144, 0.02423772932974272, -0.2671520499623872, 0.26192738874853455, -0.2838883984454118, 0.179065986029245, -0.2647794524287589, 4.878774934525946e-05, 0.1865741682719344, 0.03091746833

In [18]:
decoded_parameters = json.loads(encoded_parameters)
cls_global = LogisticRegression()

cls_global.coef_ = np.array(decoded_parameters[0])
cls_global.intercept_ = np.array(decoded_parameters[1])
cls_global.classes_ = np.array(decoded_parameters[2])

In [19]:
predictions = cls_global.predict(X_test);
result = pd.DataFrame(np.vstack((predictions, y_test)).T,columns=['Predicted Outcomes','Actual Outcomes'])
result

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


,Predicted Outcomes,Actual Outcomes
0,malicious,malicious
1,non-malicious,non-malicious
2,non-malicious,non-malicious
3,malicious,malicious
4,non-malicious,non-malicious


In [21]:
accuracy = np.sum(predictions == y_test) / y_test.shape[0] * 100
conf_matrix = confusion_matrix(predictions, y_test)
precision = conf_matrix[0,0] / (conf_matrix[0,0] + conf_matrix[0,1]) * 100

print(conf_matrix)
print("Accuracy: {0:.6f}%".format(accuracy))
print("Precision: {0:.6f}%".format(precision))

[[57  1]
 [ 1 16]]
Accuracy: 97.333333%
Precision: 98.275862%
